In [43]:
sample_text1 = "My name is Inigo Montoya. You killed my father. Prepare to die."

In [177]:
sample_text2 = "One fish, Two fish, Red fish, Blue fish, Black fish, Blue fish, Old fish, New fish. This one has a little car. This one has a little star."

In [271]:
sample_text3 = "One fish, Two fish, Red fish, Blue fish,"

Convert sentence to lower case and make it a list

In [227]:
def convert_line(sentence):
    '''
    Turns string to lower case, splits to list, returns list
    Params: sentence (string)
    Return: word_list (list)
    '''

    import re
    
    #word_list = sentence.strip(whitespace+punctuation).lower().split()
    word_list = re.sub(r'[^a-zA-Z ]', '', sentence).lower().split()
    #word_list = lower.split()
    
    return word_list

In [228]:
convert_line(sample_text2)

['one',
 'fish',
 'two',
 'fish',
 'red',
 'fish',
 'blue',
 'fish',
 'black',
 'fish',
 'blue',
 'fish',
 'old',
 'fish',
 'new',
 'fish',
 'this',
 'one',
 'has',
 'a',
 'little',
 'car',
 'this',
 'one',
 'has',
 'a',
 'little',
 'star']

Add start and end states to word list. Could be combined with convert_line()

In [149]:
def add_states(sentence):
    '''
    Adds start and end states to the word list.
    Params: orig_list (list)
    Return: orig_list (list)
    '''
    
    sentence.insert(0, "*S*")
    sentence.append("*N*")

    return sentence

In [150]:
add_states(convert_line(sample_text1))

['*S*',
 'my',
 'name',
 'is',
 'inigo',
 'montoya.',
 'you',
 'killed',
 'my',
 'father.',
 'prepare',
 'to',
 'die.',
 '*N*']

In [109]:
sample_dict1 = {
    '*S*': {'one': 1},
    'one': {'fish': 1},
    'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1},
    'two': {'fish': 1},
    'red': {'fish': 1},
    'blue': {'fish': 1}
}

In [115]:
sample_dict2 = {
    '*S*': {'one': 1},
    'one': {'fish': 1},
    'fish': {'two': 1, 'red': 2, 'blue': 3, '*E*': 4},
    'two': {'fish': 1},
    'red': {'fish': 1},
    'blue': {'fish': 1}
}

Create new dictionary with normalized probabilties. See choose_next() function to bypass normalization.

In [110]:
def normalize(orig_dict):
    '''
    Takes dictionary and creates a new dictionary where the numbers are turned into normalized transition state probabilities
    Params: orig_dict (dictionary)
    Return: norm_dict (dictionary)
    '''

    # Required to make a non-referential copy of a dictionary
    import copy
    
    #Copy dictionary
    norm_dict = copy.deepcopy(orig_dict)
    print("Norm dict: ", norm_dict)

    #orig_dict{prev_words: {curr_word:obs} )
    for prev, curr in norm_dict.items():
        #print(prev, curr)
        count = 0
        #loop through to count all observations
        for obs in curr:
            count += curr[obs]
        #loop again to adjust observations
        for obs in curr:
            curr[obs] = curr[obs] / count


        #print("Normalized to ", y)
    print("Original dict: ", orig_dict)
    print("Norm dict: ", norm_dict)
    return norm_dict


In [111]:
print("Sample dict before: ", sample_dict1)
normalized = normalize(sample_dict1)
print("Sample dict after: ", sample_dict1)

Sample dict before:  {'*S*': {'one': 1}, 'one': {'fish': 1}, 'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1}, 'two': {'fish': 1}, 'red': {'fish': 1}, 'blue': {'fish': 1}}
Norm dict:  {'*S*': {'one': 1}, 'one': {'fish': 1}, 'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1}, 'two': {'fish': 1}, 'red': {'fish': 1}, 'blue': {'fish': 1}}
Original dict:  {'*S*': {'one': 1}, 'one': {'fish': 1}, 'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1}, 'two': {'fish': 1}, 'red': {'fish': 1}, 'blue': {'fish': 1}}
Norm dict:  {'*S*': {'one': 1.0}, 'one': {'fish': 1.0}, 'fish': {'two': 0.25, 'red': 0.25, 'blue': 0.25, '*E*': 0.25}, 'two': {'fish': 1.0}, 'red': {'fish': 1.0}, 'blue': {'fish': 1.0}}
Sample dict after:  {'*S*': {'one': 1}, 'one': {'fish': 1}, 'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1}, 'two': {'fish': 1}, 'red': {'fish': 1}, 'blue': {'fish': 1}}


Choose next word based on probability. Does not require normalization function.

In [128]:
def choose_next(curr_word):
    '''
    Choose next word from available choices. Current word will be passed as single dictionary entry.
    Function will choose next word based on the values in that entry.
    Params: curr_word (dictionary)
    Return: next_word (string)
    '''
    import random
    #for word, obs in curr_word.items():
    #    print(word, obs)
    #print(curr_word.keys())
    #print(curr_word.values())

    # k = number of options to return. Only use for testing.
    next_word = random.choices(list(curr_word.keys()), weights=curr_word.values(), k=1)

    return next_word

In [144]:
choose_next(sample_dict1["fish"])

['*E*']

In [ ]:
Driver function to test training helper functions

In [229]:
def test_train():
    '''
    Driver function to test training helper functions
    Params: none
    Returns: none
    '''
    from pathlib import Path
    from pprint import pprint
    
    filename = Path("project02/data/one_fish_two_fish.txt")

    #initialize empty model
    markov_model = {}
    
    with open(filename, "r") as infile:
        # Read file line by line (not using readlines() as vcf files can be large)
        for line in infile:
            fixed_line = add_states(convert_line(line))
            #print(fixed_line)
            
            markov_model = build_markov_model(markov_model, fixed_line)
    
    pprint(markov_model)
        

In [230]:
test_train()

Added one to *S*
Added fish to one
Added two to fish
Added fish to two
Added red to fish
Added fish to red
Added blue to fish
Added fish to blue
Added *N* to fish
Added black to *S*
Added fish to black
Added blue to fish
Added old to fish
Added fish to old
Added new to fish
Added fish to new
Added *N* to fish
Added this to *S*
Added one to this
Added has to one
Added a to has
Added little to a
Added car to little
Added *N* to car
Added star to little
Added *N* to star
Added say to *S*
Added what to say
Added a to what
Added lot to a
Added of to lot
Added fish to of
Added there to fish
Added are to there
Added *N* to are
Added yes to *S*
Added some to yes
Added are to some
Added red to are
Added and to red
Added some to and
Added blue to are
Added *N* to blue
Added some to *S*
Added old to are
Added and to old
Added new to are
Added *N* to new
Added sad to are
Added and to sad
Added glad to are
Added *N* to glad
Added and to *S*
Added very to are
Added very to very
Added bad to very
Add

In [286]:
def build_markov_model(markov_model, new_text):
    '''
    Test build
    Params: markov_model (dict) : existing or empty markov_model
            new_text (list) : new text to add to model
    Return: updated_model (dict) : model updated with new text
    '''
    from pprint import pprint
    
    # for each word, add to dictionary
    # automatically stops when all words are exhausted and [word, *E*] are last two entries in list
    
    while len(new_text) > 1:
        print("Checking", new_text[0], ":", new_text[1])
        # current word not in dictionary
        if new_text[0] not in markov_model:            
            markov_model[new_text[0]] = [{new_text[1] : 1}]
            print("Added", markov_model[new_text[0]])
        
        # current word in dictionary, but not next word 
        elif new_text[1] not in markov_model[new_text[0]]:
            print(new_text[1], "is ot in", markov_model[new_text[0]])
            markov_model[new_text[0]] = markov_model[new_text[0]].append([{new_text[1] : 1}])
            print("Added", markov_model[new_text[0]])
            #else:
            #    if new_text[1] not in markov_model[curr]:
            #        markov_model.update({new_text[1] : 1}}
            #        print(new_text[1], "added")
        else:
            #print("Updating existing entry")
            markov_model[new_text[0]][new_text[1]] += 1
            
        new_text.pop(0)
    return(markov_model)
                
    

In [285]:
from pprint import pprint

markov_model = {}
new_text = add_states(convert_line(sample_text3))
print(sample_text3)
pprint(build_markov_model(markov_model, new_text))


One fish, Two fish, Red fish, Blue fish,
Checking *S* : one
Added [{'one': 1}]
Checking one : fish
Added [{'fish': 1}]
Checking fish : two
Added [{'two': 1}]
Checking two : fish
Added [{'fish': 1}]
Checking fish : red
red is ot in [{'two': 1}]
Added None
Checking red : fish
Added [{'fish': 1}]
Checking fish : blue


TypeError: argument of type 'NoneType' is not iterable